In [ ]:
import sys
import os

# 1. Get the path to the current script (examples/your_script.py)
# 2. Go one level up to get the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# 3. Add that root to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.camera_detector import CameraDetector
from src.lidar_detector import LiDARDetector
from src.radar_detector import RadarDetector
from src.fusion_engine import FusionEngine

# Single Modality Detection
camera_detector = CameraDetector()
camera_detections = camera_detector.detect("/Users/hamidrezamatiny/Documents/GitHub/Auto_driving/examples/sample_data/sample_image.jpg")

# Print results
camera_detector.print_detections(camera_detections)

# Visualize
camera_detector.visualize_results(camera_detections)

100%|██████████| 6.23M/6.23M [00:00<00:00, 8.34MB/s]



Detection Results - 8 objects found

[1] CAR
    Confidence: 82.19%
    bbox_2d: [857, 252, 1018, 362]
    class_id: 2
    bbox_center: [937, 307]
    bbox_width: 161
    bbox_height: 110

[2] CAR
    Confidence: 76.48%
    bbox_2d: [630, 229, 689, 282]
    class_id: 2
    bbox_center: [659, 255]
    bbox_width: 59
    bbox_height: 53

[3] CAR
    Confidence: 74.76%
    bbox_2d: [275, 242, 469, 385]
    class_id: 2
    bbox_center: [372, 313]
    bbox_width: 194
    bbox_height: 143

[4] CAR
    Confidence: 69.71%
    bbox_2d: [134, 242, 177, 263]
    class_id: 2
    bbox_center: [155, 252]
    bbox_width: 43
    bbox_height: 21

[5] CAR
    Confidence: 65.12%
    bbox_2d: [844, 241, 894, 285]
    class_id: 2
    bbox_center: [869, 263]
    bbox_width: 50
    bbox_height: 44

[6] CAR
    Confidence: 63.41%
    bbox_2d: [67, 236, 122, 270]
    class_id: 2
    bbox_center: [94, 253]
    bbox_width: 55
    bbox_height: 34

[7] CAR
    Confidence: 62.14%
    bbox_2d: [746, 236, 814, 291]


: 

In [6]:
import sys
import os
import json
import numpy as np
from nuscenes.utils.data_classes import RadarPointCloud

# 1. Path setup: Add the project root to sys.path so we can import from 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now we can import the detector
from src.radar_detector import RadarDetector


def main():
    # 1. Path to the binary file
    radar_data_path = "/Users/hamidrezamatiny/Downloads/v1.0-mini/samples/RADAR_FRONT_LEFT/n008-2018-08-27-11-48-51-0400__RADAR_FRONT_LEFT__1535385093657127.pcd"

    # 2. Use the NuScenes DevKit to load the binary data
    # This converts the binary PCD into a format your detector can understand
    pc = RadarPointCloud.from_file(radar_data_path)
    
    # NuScenes Radar PC structure: [x, y, z, dyn_prop, id, rcs, vx, vy, vx_comp, vy_comp, is_quality_valid, ... ]
    # We convert this into a list of dicts that your detector expects
    points = pc.points.T
    formatted_detections = []
    for p in points:
        formatted_detections.append({
            'range': float(np.sqrt(p[0]**2 + p[1]**2)),
            'azimuth': float(np.degrees(np.arctan2(p[1], p[0]))),
            'doppler_velocity': float(p[8]), # vx_comp (compensated velocity)
            'rcs': float(p[5]),
            'snr': 20.0, # Defaulting SNR as it's not always in raw PCD
            'id': int(p[4])
        })

    # 3. Initialize detector
    detector = RadarDetector()
    
    # 4. Instead of detector.detect(path), we manually set the data
    # Or modify your detector.detect to accept a list directly
    detections = []
    for det_data in formatted_detections:
        parsed = detector._parse_radar_detection(det_data)
        if parsed:
            detections.append(parsed)

    # 5. Visualize
    detector.visualize_results(detections)

if __name__ == "__main__":
    main()


RADAR DETECTION RESULTS - 9 objects detected
ID   Class           Range      Azimuth    Velocity     RCS        SNR      Conf    
--------------------------------------------------------------------------------
1    stationary_object 6.61       2.60       0.03         -0.50      20.00    90.00%  
2    stationary_object 11.10      43.91      -0.08        -0.50      20.00    90.00%  
3    stationary_object 11.18      -56.31     -0.08        10.50      20.00    90.00%  
4    stationary_object 15.20      44.73      0.05         1.00       20.00    90.00%  
5    stationary_object 16.37      48.72      0.05         0.50       20.00    90.00%  
6    stationary_object 17.92      52.94      0.11         19.00      20.00    90.00%  
7    stationary_object 20.06      57.42      0.01         7.00       20.00    90.00%  
8    stationary_object 6.42       4.47       0.01         -1.00      20.00    90.00%  
9    stationary_object 10.40      -1.65      0.15         8.00       20.00    90.00%  



In [8]:
import sys
import os
import numpy as np
from nuscenes.utils.data_classes import LidarPointCloud


# Import the detector
from src.lidar_detector import LiDARDetector

def main():
    # 2. Path to the real binary LiDAR file
    lidar_data_path = "/Users/hamidrezamatiny/Downloads/v1.0-mini/samples/LIDAR_TOP/n015-2018-10-08-15-36-50+0800__LIDAR_TOP__1538984239948316.pcd.bin"

    print("--- Starting LiDAR Detection ---")

    # 3. Load the binary PCD file using NuScenes tools
    # Raw LiDAR PCD is binary and contains [x, y, z, intensity, ring_index]
    pc = LidarPointCloud.from_file(lidar_data_path)
    
    # We only need the first 3 rows (x, y, z) for the detector
    # Transpose it so it's an Nx3 array
    points = pc.points[:3, :].T

    # 4. Initialize the LiDAR Detector
    # voxel_size: 0.5 meters (larger is faster, smaller is more precise)
    # min_points: 10 (ignore small noise clusters)
    detector = LiDARDetector(voxel_size=0.5, min_points=10)

    # 5. Inject the points and run detection
    # We bypass the file-loading internal logic since we pre-loaded the binary
    detector.pointcloud = points
    clusters = detector._cluster_points(points)
    
    detections = []
    for i, cluster in enumerate(clusters):
        det = detector._cluster_to_detection(cluster, i)
        if det:
            detections.append(det)

    # 6. Display Results
    if detections:
        print(f"Successfully detected {len(detections)} objects!")
        print(f"{'ID':<4} {'Class':<15} {'Distance':<10} {'Points':<8} {'Confidence':<10}")
        print("-" * 55)
        
        for i, det in enumerate(detections):
            # Getting distance from the Detection object attributes
            dist = det.attributes.get('distance', 0)
            pts = det.attributes.get('num_points', 0)
            print(f"{i:<4} {det.class_name:<15} {dist:<10.2f} {pts:<8} {det.confidence:<10.2%}")
        
        # 7. Visualization
        # This will attempt to open an Open3D window if installed
        # Otherwise, it will print a confirmation message.
        detector.visualize_results(detections)
    else:
        print("No objects found. Try decreasing 'voxel_size' or 'min_points'.")

if __name__ == "__main__":
    main()

--- Starting LiDAR Detection ---
Successfully detected 193 objects!
ID   Class           Distance   Points   Confidence
-------------------------------------------------------
0    car             5.27       15875    70.00%    
1    pedestrian      38.41      27       85.00%    
2    object          0.28       8047     60.00%    
3    object          30.81      11       60.00%    
4    object          14.50      31       60.00%    
5    car             14.78      154      70.00%    
6    object          15.52      14       60.00%    
7    pedestrian      36.75      15       85.00%    
8    object          35.45      10       60.00%    
9    object          30.92      14       60.00%    
10   car             33.47      48       70.00%    
11   car             35.09      20       70.00%    
12   object          34.23      10       60.00%    
13   object          14.86      14       60.00%    
14   object          24.97      11       60.00%    
15   truck           22.70      25       88.